# 05: Image statistics of all processed images

Measures every processed image of EyePACS, APTOS, IDRiD and DDR (fundus crop at 512 px, before
local mean normalisation), inside the fundus only: brightness and contrast of the luminance,
mean of each colour channel, share of under- and over-exposed pixels, sharpness (variance of
the Laplacian) and how much of the fundus disc is inside the camera frame. These describe how
acquisition differs between sites, and serve later as the reference for drift monitoring.

**Settings** (right-hand panel): Accelerator **None**, Internet **On** (to install the package).
**Inputs** (*Add Input*):
- *Notebooks* → *Your Work* → **01: Preprocess EyePACS**, **02: Preprocess APTOS**,
  **03: Preprocess IDRiD**, **04: Preprocess DDR**

**Run:** *Save Version* → *Save & Run All (Commit)*. Takes about 15–25 min.

**Output** (`/kaggle/working/image_stats/`):
- `image_stats.parquet`: one row per processed image
- `summary.json`, `stats_by_source.png`

> RetinaGuard is a research prototype, **not a medical device**.

In [ ]:
REPO_URL = "https://github.com/FidaMarzouk/retinaguard.git"
GIT_REF = "main"
OUT_DIR = "/kaggle/working/image_stats"
SOURCES = ("eyepacs", "aptos", "idrid", "ddr")

In [ ]:
%pip install -q "retinaguard @ git+{REPO_URL}@{GIT_REF}" "pandera>=0.20"

In [ ]:
# --- 1. Find the processed images of every source ------------------------------
import subprocess
from pathlib import Path

import pandas as pd

from retinaguard import __version__
from retinaguard.data.processing import shard_roots, split_processed_path

INPUT_ROOT = Path("/kaggle/input")
manifests, roots = [], {}
for source in SOURCES:
    found = sorted(INPUT_ROOT.glob(f"**/manifest_{source}.parquet"))
    assert found, f"manifest_{source}.parquet not found: attach the output of its notebook"
    manifests.append(pd.read_parquet(found[0]))
    roots.update(shard_roots(found[0].parent))
manifest = pd.concat(manifests, ignore_index=True).dropna(subset=["processed_path"])
shard_of = manifest["processed_path"].map(lambda p: split_processed_path(p)[0])
missing = sorted(set(shard_of) - set(roots))
assert not missing, f"shards not found: {missing[:5]}"

git_sha = subprocess.run(
    ["git", "ls-remote", REPO_URL, GIT_REF], capture_output=True, text=True
).stdout.split("\t")[0]
print(f"retinaguard {__version__} @ {git_sha[:10]}")
print(f"{len(manifest):,} processed images in {shard_of.nunique()} shards")
print(manifest["source"].value_counts().to_dict())

In [ ]:
# --- 2. Measure every image, one shard per worker -------------------------------
import os
import time
import zipfile
from concurrent.futures import ProcessPoolExecutor

from retinaguard.data.image_stats import STATS_VERSION, image_stats
from retinaguard.data.preprocess import decode_rgb

jobs = {
    shard: list(zip(rows["image_id"], rows["processed_path"], strict=True))
    for shard, rows in manifest.groupby(shard_of)
}


def measure_shard(shard):
    rows = []
    with zipfile.ZipFile(roots[shard] / shard) as archive:
        for image_id, path in jobs[shard]:
            img = decode_rgb(archive.read(split_processed_path(path)[1]))
            rows.append({"image_id": image_id, **image_stats(img)})
    return rows


start = time.perf_counter()
rows = []
with ProcessPoolExecutor(os.cpu_count()) as pool:
    for i, shard_rows in enumerate(pool.map(measure_shard, sorted(jobs)), start=1):
        rows.extend(shard_rows)
        if i % 5 == 0 or i == len(jobs):
            print(f"{i}/{len(jobs)} shards, {len(rows):,} images", flush=True)
stats = pd.DataFrame(rows).assign(stats_version=STATS_VERSION)
assert len(stats) == len(manifest)
assert stats["image_id"].is_unique
minutes = (time.perf_counter() - start) / 60
print(f"{minutes:.1f} min")

In [ ]:
# --- 3. Save and summarise per source -------------------------------------------
import json

from retinaguard.data.image_stats import STAT_NAMES

out_dir = Path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)
stats.to_parquet(out_dir / "image_stats.parquet", index=False)

by_source = stats.merge(manifest[["image_id", "source"]], on="image_id")
quantiles = by_source.groupby("source")[list(STAT_NAMES)].quantile([0.1, 0.5, 0.9])
summary = {
    "retinaguard_version": __version__,
    "git_sha": git_sha,
    "stats_version": STATS_VERSION,
    "images": by_source["source"].value_counts().to_dict(),
    "minutes": round(minutes, 1),
    "p10_p50_p90": {
        source: {
            name: [round(float(v), 3) for v in quantiles.loc[source, name]] for name in STAT_NAMES
        }
        for source in SOURCES
    },
}
_ = (out_dir / "summary.json").write_text(json.dumps(summary, indent=2))
pd.set_option("display.width", 200)
print(quantiles.xs(0.5, level=1).round(3).loc[list(SOURCES)].to_string())

In [ ]:
# --- 4. Distributions per source ------------------------------------------------
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 3, figsize=(12, 9))
for ax, name in zip(axes.flat, STAT_NAMES, strict=True):
    ax.boxplot(
        [by_source.loc[by_source["source"] == s, name] for s in SOURCES],
        tick_labels=SOURCES,
        showfliers=False,
    )
    ax.set_title(name, fontsize=10)
plt.tight_layout()
fig.savefig(out_dir / "stats_by_source.png", dpi=80)
plt.show()

**After the version finishes:** download `image_stats/image_stats.parquet` from the **Output**
tab into `data/interim/image_stats/`.